In [1]:
from cropspectai import config
from cropspectai.modeling.predict import predict_model
from cropspectai.data_analysis import feature_engineering
import pandas as pd


# --------------------------------------------------
# Load data
# --------------------------------------------------


X_new = pd.read_csv(config.BASIL_EXPERIMENT_DATA_PATH / "X_new.csv", index_col = False)
print("Shape of imported data: ", X_new.shape)

Shape of imported data:  (24, 15)


In [2]:
column_list = X_new.columns.to_list()
print(column_list)

['timestamp', 'specimen', 'noisy_channels', 'F1', 'F2', 'FZ', 'F3', 'F4', 'FY', 'F5', 'FXL', 'F6', 'F7', 'F8', 'NIR']


In [3]:
AS7343_CHANNELS = list(config.AS7343_CHANNELS.keys())
print(AS7343_CHANNELS)

['F1', 'F2', 'FZ', 'F3', 'F4', 'FY', 'F5', 'FXL', 'F6', 'F7', 'F8', 'NIR']


In [4]:
non_features = column_list[:3]
print(non_features)
X_non_features = X_new[non_features].copy()
print(X_non_features.shape)

['timestamp', 'specimen', 'noisy_channels']
(24, 3)


In [5]:
X_new = X_new[AS7343_CHANNELS].copy()


In [6]:
X_new = feature_engineering.add_all_features(X_new).copy()
print(X_new.shape)

(24, 40)


In [7]:
X_new.columns

Index(['F1', 'F2', 'FZ', 'F3', 'F4', 'FY', 'F5', 'FXL', 'F6', 'F7', 'F8',
       'NIR', 'ci_red_edge_as7343', 'ci_green_as7343', 'gm_nd_as7343',
       'red_edge_ratio_as7343', 'datt2_as7343', 'datt1_as7343', 'mcari_as7343',
       'tcari_as7343', 'osavi_as7343', 'mcari_osavi_as7343',
       'tcari_osavi_as7343', 'mndvi_as7343', 'ndvi_as7343', 'gndvi_as7343',
       'sr_as7343', 'green_sr_as7343', 'evi2_as7343', 'rdvi_as7343',
       'sipi_as7343', 'psri_as7343', 'red_green_nd', 'red_edge_green_nd',
       'nir_green_nd', 'red_edge_slope_proxy', 'green_red_ratio',
       'rededge_green_ratio', 'nir_fy_ratio', 'nir_fxl_ratio'],
      dtype='str')

## import model config json file for feature list

In [8]:
import json

model_config_path = config.MODELS_DIR / "xgb_best_parameters_v_01.json"
with open(model_config_path, "r") as f:
    model_config = json.load(f)
features = model_config["features"]
print(features)

['F1', 'F2', 'FZ', 'F3', 'F4', 'FY', 'F5', 'FXL', 'F6', 'F7', 'F8', 'NIR', 'mndvi_as7343', 'sipi_as7343', 'ci_red_edge_as7343', 'psri_as7343', 'red_green_nd', 'datt1_as7343', 'mcari_osavi_as7343']


### remove columns not in feature list

In [9]:
X_new = X_new[features].copy()
X_new.columns

Index(['F1', 'F2', 'FZ', 'F3', 'F4', 'FY', 'F5', 'FXL', 'F6', 'F7', 'F8',
       'NIR', 'mndvi_as7343', 'sipi_as7343', 'ci_red_edge_as7343',
       'psri_as7343', 'red_green_nd', 'datt1_as7343', 'mcari_osavi_as7343'],
      dtype='str')

## load best xgb model

In [10]:
model_path = config.MODELS_DIR / "xgb_v_01.joblib"

In [11]:
predictions = predict_model(X_new, model_path)
print(predictions)

[19.325014 22.56456  25.53629  23.261965 24.660383 22.603994 21.81295
 21.640808 49.590717 37.080364 23.773594 24.58659  23.75407  23.797398
 18.981905 18.053276 22.039793 22.039793 17.899874 17.700214 20.414234
 17.783699 17.654444 18.9075  ]


In [12]:
df_predict = X_non_features[non_features].copy()
df_predict["predictions"] = predictions

In [13]:
df_predict

,timestamp,specimen,noisy_channels,predictions
0,2026-10-10T10:57:57,sun_water,F3 (129.1%),19.325014
1,2026-10-10T10:58:19,sun_water,FZ (14.3%),22.564560
2,2026-10-10T10:58:52,sun_water,F3 (9.6%); NIR (28.4%),25.536289
3,2026-10-10T11:00:51,sun_no_water,FZ (6.5%); F8 (5.4%),23.261965
4,2026-10-10T11:01:20,sun_no_water,NaN,24.660383
5,2026-10-10T11:02:06,sun_no_water,F2 (86.1%),22.603994
6,2026-10-10T11:04:17,dark_water,NaN,21.812950
7,2026-10-10T11:04:32,dark_water,NaN,21.640808
8,2026-10-10T11:04:55,dark_water,F2 (69.0%); F3 (10.9%); F7 (7.0%),49.590717
9,2026-10-10T11:05:35,dark_no_water,NaN,37.080364
